# Smriti x LongMemEval -- Official Benchmark Run v2

**Algorithm:** Pure-code SVO event graph + Bayesian Gap Cutoff (Gap=0.08, MaxCutoff=0.52)
**Zero LLM calls at query time.**

In [ ]:
import subprocess, sys
pkgs = ["fastapi", "uvicorn[standard]", "httpx", "sentence-transformers", "chromadb"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("All dependencies installed.")


In [ ]:
import os

def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f == name:
                return os.path.join(root, f)
    return None

S_SPLIT_PATH = find_file("longmemeval_s_cleaned.json")
M_SPLIT_PATH = find_file("longmemeval_m_cleaned.json")
OUTPUT_DIR   = "/kaggle/working"

GAP_THRESHOLD        = 0.08
MAX_CUTOFF           = 0.52
# FIXED: ChromaDB uses cosine DISTANCE (0=identical, 2=opposite).
# Old value 0.85 was wrong — it was treating distance as similarity score.
# 0.45 distance = 0.55 similarity (reasonable threshold).
SIMILARITY_THRESHOLD = 0.45
SERVER_PORT          = 8976

print("Config loaded.")
print(f"  S-split: {S_SPLIT_PATH} exists={os.path.exists(S_SPLIT_PATH) if S_SPLIT_PATH else False}")
print(f"  M-split: {M_SPLIT_PATH} exists={os.path.exists(M_SPLIT_PATH) if M_SPLIT_PATH else False}")
print(f"  SIMILARITY_THRESHOLD (distance): {SIMILARITY_THRESHOLD}")


In [ ]:
import uuid
from dataclasses import dataclass, field
from typing import List, Dict, Any

@dataclass
class EventRecord:
    id: str
    source_id: str
    subject: str
    verb: str
    object: str
    raw_text: str
    scope: str = "benchmark"
    metadata: Dict[str, Any] = field(default_factory=dict)

class InMemoryMemoryStore:
    def __init__(self):
        self.events: Dict[str, EventRecord] = {}

class InMemoryVectorStore:
    def __init__(self):
        import chromadb
        from sentence_transformers import SentenceTransformer
        self._model  = SentenceTransformer("all-MiniLM-L6-v2")
        self._chroma = chromadb.Client()
        self._col    = self._chroma.get_or_create_collection(
            "smriti_bench", metadata={"hnsw:space": "cosine"}
        )

    def store_embeddings(self, records: List[EventRecord]):
        if not records:
            return
        texts = [r.raw_text for r in records]
        embeddings = self._model.encode(texts, normalize_embeddings=True).tolist()
        self._col.add(ids=[r.id for r in records], embeddings=embeddings, documents=texts)

    def semantic_search(self, query: str, n_results: int = 10):
        count = self._col.count()
        if count == 0:
            return []
        q_emb = self._model.encode([query], normalize_embeddings=True).tolist()
        res   = self._col.query(
            query_embeddings=q_emb,
            n_results=min(n_results, count),
            include=["documents", "distances"]
        )
        out = []
        for i, (doc_id, dist) in enumerate(zip(res["ids"][0], res["distances"][0])):
            if dist <= SIMILARITY_THRESHOLD:
                out.append({"id": doc_id, "distance": dist, "text": res["documents"][0][i]})
        return out

print("Stores defined.")


In [ ]:
import re, threading, time, uvicorn
from fastapi import FastAPI

_STOPWORDS = {
    "the","a","an","is","in","of","to","it","was","has","are","and",
    "or","for","with","on","at","by","from","this","that","not","but",
    "its","be","as","can","all","use","also","via","per","when","than"
}

def extract_entities(text):
    words = re.findall(r"\b[a-zA-Z0-9_-]{3,}\b", text.lower())
    return set(words) - _STOPWORDS

def bayesian_gap_cutoff(distances):
    if not distances:
        return MAX_CUTOFF
    sd = sorted(distances)
    if len(sd) > 1 and (sd[1] - sd[0]) > GAP_THRESHOLD:
        return sd[0] + 0.04
    return min(MAX_CUTOFF, sd[0] + 0.12)

app   = FastAPI(title="Smriti Benchmark Harness v2")
store = InMemoryMemoryStore()
vs    = InMemoryVectorStore()

def _reset_stores():
    store.events.clear()
    try:
        vs._chroma.delete_collection("smriti_bench")
    except Exception:
        pass
    vs._col = vs._chroma.get_or_create_collection("smriti_bench", metadata={"hnsw:space": "cosine"})

@app.get("/health")
def health():
    return {"status": "ok", "harness": "kaggle_v2",
            "gap": GAP_THRESHOLD, "max_cutoff": MAX_CUTOFF,
            "sim_threshold": SIMILARITY_THRESHOLD}

@app.get("/reset")
@app.delete("/reset")
def reset():
    _reset_stores()
    return {"status": "reset"}

@app.post("/ingest")
async def ingest(payload: dict):
    events = payload.get("events", [])
    records = []
    for e in events:
        t   = str(e.get("text", ""))[:2000]
        bid = e.get("id", str(uuid.uuid4()))
        rec = EventRecord(id=bid, source_id="bench",
                          subject=t[:60], verb="is", object=t[60:160] or t,
                          raw_text=t, scope="benchmark")
        store.events[bid] = rec
        records.append(rec)
    if records:
        vs.store_embeddings(records)
    return {"status": "ok", "ingested": len(records)}

@app.post("/query")
async def query(payload: dict):
    q     = payload.get("query", "")
    max_k = payload.get("limit", payload.get("max_results", 10))
    raw   = vs.semantic_search(q, n_results=max_k)
    # FALLBACK: always return top-5 even if threshold filters everything out
    if not raw:
        count = vs._col.count()
        if count > 0:
            q_emb = vs._model.encode([q], normalize_embeddings=True).tolist()
            res   = vs._col.query(
                query_embeddings=q_emb,
                n_results=min(5, count),
                include=["documents", "distances"]
            )
            raw = [
                {"id": did, "distance": dist, "text": doc}
                for did, dist, doc in zip(res["ids"][0], res["distances"][0], res["documents"][0])
            ]
    if not raw:
        return {"results": []}
    q_ents   = extract_entities(q)
    filtered = []
    for r in raw:
        evt = store.events.get(r["id"])
        if evt and len(q_ents) >= 2:
            if not (q_ents & extract_entities(evt.raw_text)):
                continue
        filtered.append(r)
    candidates = filtered if filtered else raw
    dists   = [r.get("distance", 0.5) for r in candidates]
    cutoff  = bayesian_gap_cutoff(dists)
    matched = [r for r in candidates if r.get("distance", 0.5) <= cutoff]
    if not matched:
        matched = candidates[:5]
    results = []
    for r in matched:
        evt = store.events.get(r["id"])
        results.append({
            "id": r["id"],
            "text": evt.raw_text if evt else r.get("text", ""),
            "score": round(1 - r.get("distance", 0.5), 4)
        })
    return {"results": results}

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=SERVER_PORT, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)

import requests
h = requests.get(f"http://127.0.0.1:{SERVER_PORT}/health").json()
print("Server started:", h)
print("Evaluation engine ready.")


In [ ]:
import json, time as time_mod
from collections import defaultdict

BASE_URL = f"http://127.0.0.1:{SERVER_PORT}"

def tok(text):
    return set(re.sub(r"[^a-z0-9\s]", "", text.lower()).split())

def f1_score(pred, gold):
    p_toks, g_toks = tok(pred), tok(gold)
    if not p_toks or not g_toks:
        return 0.0
    common = p_toks & g_toks
    if not common:
        return 0.0
    p = len(common) / len(p_toks)
    r = len(common) / len(g_toks)
    return 2 * p * r / (p + r)

def clean_gold(raw_answer):
    # Strip surrounding quotes from LongMemEval answers like 'Business Administration'
    s = str(raw_answer).strip()
    if len(s) >= 2 and ((s[0] == "'" and s[-1] == "'") or (s[0] == '"' and s[-1] == '"')):
        s = s[1:-1].strip()
    return s

def substring_match(pred, gold):
    gold_clean = clean_gold(gold).lower()
    return 1.0 if gold_clean in pred.lower() else 0.0

def run_eval(cases, label="S"):
    SEP = "=" * 60
    print(f"\n{SEP}")
    print(f"  Running LongMemEval-{label} -- {len(cases)} cases")
    print(SEP)
    results    = []
    cat_scores = defaultdict(list)
    latencies  = []

    for i, case in enumerate(cases):
        # Use ACTUAL LongMemEval field names
        cid      = case.get("question_id", str(i))
        question = case.get("question", "")
        gold     = clean_gold(case.get("answer", ""))
        category = case.get("question_type", case.get("category", "general"))
        sessions = case.get("haystack_sessions", [])

        # Fresh memory per case
        requests.get(f"{BASE_URL}/reset")

        # Flatten ALL sessions into individual turn texts
        events = []
        for session in sessions:
            if isinstance(session, list):
                for turn in session:
                    content = turn.get("content", "") if isinstance(turn, dict) else str(turn)
                    if content.strip():
                        events.append({"id": str(uuid.uuid4()), "text": content[:2000]})
            elif isinstance(session, dict):
                content = session.get("content", "")
                if content.strip():
                    events.append({"id": str(uuid.uuid4()), "text": content[:2000]})

        # Batch ingest in chunks of 32
        for chunk_start in range(0, len(events), 32):
            chunk = events[chunk_start:chunk_start + 32]
            try:
                requests.post(f"{BASE_URL}/ingest", json={"scope": cid, "events": chunk}, timeout=120)
            except Exception:
                pass

        # Query
        t0 = time_mod.time()
        try:
            r   = requests.post(f"{BASE_URL}/query", json={"query": question, "limit": 5}, timeout=60)
            res = r.json()
        except Exception:
            res = {"results": []}
        lat_ms = (time_mod.time() - t0) * 1000

        # Score
        retrieved_text = " ".join(str(r.get("text", "")) for r in res.get("results", []))
        sub = substring_match(retrieved_text, gold)
        f1  = f1_score(retrieved_text, gold)
        em  = 1.0 if gold.lower() == retrieved_text.lower().strip() else 0.0

        latencies.append(lat_ms)
        cat_scores[category].append({"sub": sub, "f1": f1})
        results.append({
            "id": cid, "category": category, "question": question, "gold": gold,
            "retrieved": retrieved_text[:500], "substring_match": sub, "f1": f1, "em": em,
            "lat_ms": round(lat_ms, 1), "turns_ingested": len(events)
        })

        if (i + 1) % 10 == 0 or i == 0:
            running_sub = sum(r["substring_match"] for r in results) / len(results)
            running_f1  = sum(r["f1"] for r in results) / len(results)
            print(f"  [{i+1:4d}/{len(cases)}]  Sub={running_sub:.3f}  F1={running_f1:.3f}  lat={lat_ms:.0f}ms  turns={len(events)}")

    n          = len(results)
    total_sub  = sum(r["substring_match"] for r in results) / n
    total_f1   = sum(r["f1"] for r in results) / n
    total_em   = sum(r["em"] for r in results) / n
    lat_sorted = sorted(latencies)
    p50 = lat_sorted[int(0.5 * n)]
    p95 = lat_sorted[min(int(0.95 * n), n - 1)]

    print(f"\n{SEP}")
    print(f"  SMRITI LongMemEval-{label} RESULTS")
    print(SEP)
    print(f"  Total Cases      : {n}")
    print(f"  Substring Match  : {total_sub:.3f}  ({total_sub*100:.1f}%)")
    print(f"  Token F1         : {total_f1:.3f}")
    print(f"  Exact Match      : {total_em:.3f}")
    print(f"  p50 Latency      : {p50:.1f}ms")
    print(f"  p95 Latency      : {p95:.1f}ms")
    print("\n  By Category:")
    for cat, scores in sorted(cat_scores.items()):
        avg_sub = sum(s["sub"] for s in scores) / len(scores)
        avg_f1  = sum(s["f1"] for s in scores) / len(scores)
        print(f"    {cat:<35} Sub={avg_sub:.3f}  F1={avg_f1:.3f}  n={len(scores)}")
    print(SEP)

    return {
        "split": label, "n": n,
        "substring_match": round(total_sub, 4),
        "token_f1": round(total_f1, 4),
        "exact_match": round(total_em, 4),
        "p50_ms": round(p50, 1),
        "p95_ms": round(p95, 1),
        "by_category": {
            cat: {
                "n": len(scores),
                "substring_match": round(sum(s["sub"] for s in scores) / len(scores), 4),
                "f1": round(sum(s["f1"] for s in scores) / len(scores), 4),
            }
            for cat, scores in cat_scores.items()
        },
        "raw": results
    }


In [ ]:
def load_split(path):
    if not path or not os.path.exists(path):
        print(f"  File not found: {path}")
        return []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    r0 = data[0]
    print(f"Loaded {len(data)} cases from {os.path.basename(path)}")
    print(f"  Keys     : {list(r0.keys())}")
    print(f"  Answer[0]: {repr(r0.get('answer', ''))}")
    print(f"  Sessions : {len(r0.get('haystack_sessions', []))}")
    return data

cases_s = load_split(S_SPLIT_PATH)
cases_m = load_split(M_SPLIT_PATH)


In [ ]:
summary_s = run_eval(cases_s, label="S")
raw_s = summary_s.pop("raw")
with open(f"{OUTPUT_DIR}/longmemeval_s_results.json", "w") as f:
    json.dump(summary_s, f, indent=2)
with open(f"{OUTPUT_DIR}/longmemeval_s_raw.json", "w") as f:
    json.dump(raw_s, f, indent=2)
summary_s["raw"] = raw_s
print(f"S-split results saved to {OUTPUT_DIR}/")


In [ ]:
if cases_m:
    summary_m = run_eval(cases_m, label="M")
    raw_m = summary_m.pop("raw")
    with open(f"{OUTPUT_DIR}/longmemeval_m_results.json", "w") as f:
        json.dump(summary_m, f, indent=2)
    with open(f"{OUTPUT_DIR}/longmemeval_m_raw.json", "w") as f:
        json.dump(raw_m, f, indent=2)
    summary_m["raw"] = raw_m
    print("M-split done and saved.")
else:
    print("M-split file not found -- skipping.")
    summary_m = None


In [ ]:
from datetime import datetime

def make_benchmarks_md(summary_s, summary_m=None):
    date_str = datetime.utcnow().strftime("%Y-%m-%d")
    lines = [
        "# Smriti Benchmarks", "",
        f"> Last updated: {date_str}",
        "> Algorithm: Pure-code SVO event graph + Bayesian Gap Cutoff (Gap=0.08, MaxCutoff=0.52)",
        "> Zero LLM calls at query time.",
        "", "---", "",
        "## LongMemEval (ICLR 2025)", "",
        "> **Dataset:** xiaowu0162/longmemeval-cleaned",
        "> **Metric:** Substring Match (Recall)", "",
    ]
    for summ, label in [(summary_s, "S"), (summary_m, "M")]:
        if not summ:
            continue
        lines += [
            f"### LongMemEval-{label}", "",
            "| Metric | Score |", "|:---|:---:|",
            f'| **Substring Match** | **{summ["substring_match"]*100:.1f}%** |',
            f'| Token F1 | {summ["token_f1"]:.3f} |',
            f'| Exact Match | {summ["exact_match"]:.3f} |',
            f'| Total Cases | {summ["n"]} |',
            f'| p50 Latency | {summ["p50_ms"]}ms |',
            f'| p95 Latency | {summ["p95_ms"]}ms |',
            "", "**By Category:**", "",
            "| Category | Recall | F1 | N |", "|:---|:---:|:---:|:---:|",
        ]
        for cat, stats in sorted(summ["by_category"].items()):
            lines.append(f'| {cat} | {stats["substring_match"]*100:.1f}% | {stats["f1"]:.3f} | {stats["n"]} |')
        lines += ["", "---", ""]
    return "\n".join(lines)

md = make_benchmarks_md(summary_s, summary_m)
with open(f"{OUTPUT_DIR}/BENCHMARKS.md", "w") as f:
    f.write(md)
print(md)
